# Final Model Tuning
## CW1 - Best Configuration Selection & Submission

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

np.random.seed(123)

trn = pd.read_csv('../data/raw/CW1_train.csv')
tst = pd.read_csv('../data/raw/CW1_test.csv')

## 1. Preprocessing

In [ ]:
def preprocess(df):
    df = df.copy()
    
    for col in ['x', 'y', 'z']:
        df[col] = df[col].replace(0, np.nan)
        df[col] = df[col].fillna(df[col].median())
    
    cut_map = {'Fair': 0, 'Good': 1, 'Very Good': 2, 'Premium': 3, 'Ideal': 4}
    color_map = {'J': 0, 'I': 1, 'H': 2, 'G': 3, 'F': 4, 'E': 5, 'D': 6}
    clarity_map = {'I1': 0, 'SI2': 1, 'SI1': 2, 'VS2': 3, 'VS1': 4, 'VVS2': 5, 'VVS1': 6, 'IF': 7}
    
    df['cut_ord'] = df['cut'].map(cut_map)
    df['color_ord'] = df['color'].map(color_map)
    df['clarity_ord'] = df['clarity'].map(clarity_map)
    
    df['depth_sq'] = df['depth'] ** 2
    df['depth_cb'] = df['depth'] ** 3
    df['depth_x_b3'] = df['depth'] * df['b3']
    df['depth_x_b1'] = df['depth'] * df['b1']
    df['depth_x_a1'] = df['depth'] * df['a1']
    df['depth_x_a4'] = df['depth'] * df['a4']
    df['depth_x_table'] = df['depth'] * df['table']
    df['b3_x_b1'] = df['b3'] * df['b1']
    df['b3_x_a1'] = df['b3'] * df['a1']
    df['a1_x_a4'] = df['a1'] * df['a4']
    df['b1_x_a1'] = df['b1'] * df['a1']
    df['b3_sq'] = df['b3'] ** 2
    df['b1_sq'] = df['b1'] ** 2
    df['a1_sq'] = df['a1'] ** 2
    df['a4_sq'] = df['a4'] ** 2
    df['table_sq'] = df['table'] ** 2
    df['volume'] = df['x'] * df['y'] * df['z']
    df['log_carat'] = np.log1p(df['carat'])
    df['log_price'] = np.log1p(df['price'])
    df['xy_ratio'] = df['x'] / df['y'].replace(0, np.nan).fillna(df['y'].median())
    
    df = df.drop(columns=['cut', 'color', 'clarity'])
    return df

trn_proc = preprocess(trn)
tst_proc = preprocess(tst)

X_trn = trn_proc.drop(columns=['outcome'])
y_trn = trn_proc['outcome']
X_tst = tst_proc

common_cols = X_trn.columns.intersection(X_tst.columns)
X_trn = X_trn[common_cols]
X_tst = X_tst[common_cols]

print(f"Features: {X_trn.shape[1]}")

## 2. Phase 3 Results Summary

From our hyperparameter search in Phase 3:

| Config | Mean R² | Std |
|---|---|---|
| XGB Default | 0.4529 | 0.0154 |
| XGB Shallow+Slow | 0.4695 | 0.0158 |
| XGB Deep+Fast | 0.4464 | 0.0198 |
| **XGB Regularised** | **0.4764** | **0.0149** |
| XGB Medium | 0.4620 | 0.0161 |
| LGB Default | 0.4507 | 0.0166 |
| LGB Shallow+Slow | 0.4708 | 0.0144 |
| LGB Deep | 0.4372 | 0.0190 |
| **LGB Regularised** | **0.4768** | **0.0148** |
| LGB Medium | 0.4614 | 0.0147 |

Both models perform best with **regularised, shallow-tree configurations**. We now fine-tune around these winning configs.

## 3. Fine-Tuning Around Best Configs

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=123)

xgb_fine = {
    'XGB-A': {'n_estimators': 700, 'max_depth': 3, 'learning_rate': 0.015, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 2, 'gamma': 0.1, 'reg_alpha': 0.05, 'reg_lambda': 2.0},
    'XGB-B': {'n_estimators': 900, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.85, 'colsample_bytree': 0.85, 'min_child_weight': 3, 'gamma': 0.15, 'reg_alpha': 0.1, 'reg_lambda': 2.5},
    'XGB-C': {'n_estimators': 1200, 'max_depth': 3, 'learning_rate': 0.01, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_weight': 2, 'gamma': 0.05, 'reg_alpha': 0.03, 'reg_lambda': 1.5},
    'XGB-D': {'n_estimators': 700, 'max_depth': 4, 'learning_rate': 0.015, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 3, 'gamma': 0.1, 'reg_alpha': 0.1, 'reg_lambda': 3.0},
    'XGB-E': {'n_estimators': 1500, 'max_depth': 3, 'learning_rate': 0.008, 'subsample': 0.85, 'colsample_bytree': 0.85, 'min_child_weight': 2, 'gamma': 0.1, 'reg_alpha': 0.05, 'reg_lambda': 2.0},
}

print("=== XGBoost Fine-Tuning ===")
print(f"{'Config':10s}  {'Mean R²':>10s}  {'Std':>8s}")
print("-" * 32)
best_xgb_name = None
best_xgb_score = -1
for name, cfg in xgb_fine.items():
    m = xgb.XGBRegressor(**cfg, random_state=123, n_jobs=-1)
    scores = cross_val_score(m, X_trn, y_trn, cv=kf, scoring='r2', n_jobs=1)
    print(f"{name:10s}  {scores.mean():10.4f}  {scores.std():8.4f}")
    if scores.mean() > best_xgb_score:
        best_xgb_score = scores.mean()
        best_xgb_name = name
        best_xgb_cfg = cfg

print(f"\nBest: {best_xgb_name} (R²={best_xgb_score:.4f})")

In [ ]:
lgb_fine = {
    'LGB-A': {'n_estimators': 700, 'max_depth': 3, 'num_leaves': 47, 'learning_rate': 0.01, 'subsample': 0.9, 'colsample_bytree': 0.7, 'min_child_samples': 10, 'reg_alpha': 0.2, 'reg_lambda': 5.0},
    'LGB-B': {'n_estimators': 900, 'max_depth': 3, 'num_leaves': 31, 'learning_rate': 0.01, 'subsample': 0.85, 'colsample_bytree': 0.75, 'min_child_samples': 15, 'reg_alpha': 0.3, 'reg_lambda': 4.0},
    'LGB-C': {'n_estimators': 1200, 'max_depth': 3, 'num_leaves': 47, 'learning_rate': 0.008, 'subsample': 0.9, 'colsample_bytree': 0.7, 'min_child_samples': 10, 'reg_alpha': 0.15, 'reg_lambda': 6.0},
    'LGB-D': {'n_estimators': 700, 'max_depth': 4, 'num_leaves': 31, 'learning_rate': 0.01, 'subsample': 0.9, 'colsample_bytree': 0.65, 'min_child_samples': 12, 'reg_alpha': 0.25, 'reg_lambda': 5.0},
    'LGB-E': {'n_estimators': 1500, 'max_depth': 3, 'num_leaves': 47, 'learning_rate': 0.007, 'subsample': 0.85, 'colsample_bytree': 0.7, 'min_child_samples': 10, 'reg_alpha': 0.2, 'reg_lambda': 5.0},
}

print("=== LightGBM Fine-Tuning ===")
print(f"{'Config':10s}  {'Mean R²':>10s}  {'Std':>8s}")
print("-" * 32)
best_lgb_name = None
best_lgb_score = -1
for name, cfg in lgb_fine.items():
    m = lgb.LGBMRegressor(**cfg, random_state=123, n_jobs=-1, verbose=-1)
    scores = cross_val_score(m, X_trn, y_trn, cv=kf, scoring='r2', n_jobs=1)
    print(f"{name:10s}  {scores.mean():10.4f}  {scores.std():8.4f}")
    if scores.mean() > best_lgb_score:
        best_lgb_score = scores.mean()
        best_lgb_name = name
        best_lgb_cfg = cfg

print(f"\nBest: {best_lgb_name} (R²={best_lgb_score:.4f})")

## 4. Optimal Blend Weight Search

In [ ]:
xgb_best = xgb.XGBRegressor(**best_xgb_cfg, random_state=123, n_jobs=-1)
lgb_best = lgb.LGBMRegressor(**best_lgb_cfg, random_state=123, n_jobs=-1, verbose=-1)

xgb_oof = cross_val_predict(xgb_best, X_trn, y_trn, cv=kf, n_jobs=1)
lgb_oof = cross_val_predict(lgb_best, X_trn, y_trn, cv=kf, n_jobs=1)

weights = np.arange(0.0, 1.05, 0.05)
blend_scores = []

for w in weights:
    blend = w * xgb_oof + (1 - w) * lgb_oof
    blend_scores.append(r2_score(y_trn, blend))

best_w = weights[np.argmax(blend_scores)]
print(f"Optimal XGBoost weight: {best_w:.2f}")
print(f"Optimal LightGBM weight: {1 - best_w:.2f}")
print(f"Best blended R²: {max(blend_scores):.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(weights, blend_scores, 'o-', color='#2ecc71', linewidth=2)
ax.axvline(x=best_w, color='red', linestyle='--', alpha=0.7, label=f'Optimal w={best_w:.2f}')
ax.set_xlabel('XGBoost Weight')
ax.set_ylabel('Blended R²')
ax.set_title('Blend Weight Optimisation')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/blend_weight_search.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Final Model Diagnostics

In [ ]:
final_blend = best_w * xgb_oof + (1 - best_w) * lgb_oof
residuals = y_trn - final_blend

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].scatter(y_trn, final_blend, alpha=0.2, s=5, color='#3498db')
mn, mx = y_trn.min(), y_trn.max()
axes[0, 0].plot([mn, mx], [mn, mx], 'r--', linewidth=2)
axes[0, 0].set_title(f'Actual vs Predicted (R²={r2_score(y_trn, final_blend):.4f})')
axes[0, 0].set_xlabel('Actual')
axes[0, 0].set_ylabel('Predicted')

axes[0, 1].scatter(final_blend, residuals, alpha=0.2, s=5, color='#e74c3c')
axes[0, 1].axhline(y=0, color='black', linestyle='--')
axes[0, 1].set_title('Residuals vs Predicted')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Residual')

axes[1, 0].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='#2ecc71')
axes[1, 0].set_title(f'Residual Distribution (std={residuals.std():.3f})')
axes[1, 0].set_xlabel('Residual')

from scipy import stats
stats.probplot(residuals, plot=axes[1, 1])
axes[1, 1].set_title('Residual Q-Q Plot')

plt.tight_layout()
plt.savefig('../reports/figures/final_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Final OOF R²:   {r2_score(y_trn, final_blend):.4f}")
print(f"Final OOF RMSE: {np.sqrt(mean_squared_error(y_trn, final_blend)):.4f}")
print(f"Final OOF MAE:  {mean_absolute_error(y_trn, final_blend):.4f}")

## 6. Generate Final Submission

In [ ]:
xgb_best.fit(X_trn, y_trn)
lgb_best.fit(X_trn, y_trn)

yhat_xgb = xgb_best.predict(X_tst)
yhat_lgb = lgb_best.predict(X_tst)
yhat = best_w * yhat_xgb + (1 - best_w) * yhat_lgb

out = pd.DataFrame({'yhat': yhat})
out.to_csv('../CW1_submission_K23115695.csv', index=False)

print("Submission saved!")
print(f"Predictions: mean={yhat.mean():.3f}, std={yhat.std():.3f}, min={yhat.min():.3f}, max={yhat.max():.3f}")

## 7. Final Hyperparameters Summary

In [ ]:
print("=== Best XGBoost Config ===")
for k, v in best_xgb_cfg.items():
    print(f"  {k}: {v}")

print(f"\n=== Best LightGBM Config ===")
for k, v in best_lgb_cfg.items():
    print(f"  {k}: {v}")

print(f"\nBlend weights: XGBoost={best_w:.2f}, LightGBM={1-best_w:.2f}")
print(f"Final OOF R²: {r2_score(y_trn, final_blend):.4f}")